Module 01: Oceanic Data & Tensors (01_oceanic_data_tensors.ipynb)
----------------------------------------------------------------------------
Module 1 focuses on the foundational data engineering required to process oceanographic data for Marine Heatwave (MHW) forecasting using PyTorch.


In [ ]:
#Run in VS Code's integrated terminal to set up the virtual environment for this project.

# 1. Crear el entorno virtual dentro de la carpeta
python3.11 -m venv .venv

# 2. Activar el entorno
source .venv/bin/activate

# 3. Instalar las librerías necesarias
pip install "numpy<2" "torch" "matplotlib" "jupyter" "ipykernel"

python -m ipykernel install --user --name mhw-venv --display-name "Python (.venv MHW)"


Cell 1: Environment Setup & Oceanographic SST Data Generation
--------------------------------------------------------------------------
This cell imports core libraries and creates a synthetic 3D Sea Surface 
Temperature (SST) array representing daily satellite observations with an 
embedded Marine Heatwave (MHW) thermal anomaly.
Data Layout:
- Shape: (Total_Days, Latitude, Longitude)


In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for complete experiment reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Grid dimensions representing spatial ocean region
num_days = 30        # Temporal sequence duration (days)
lat_grid = 32        # Spatial grid height (Latitude resolution)
lon_grid = 32        # Spatial grid width (Longitude resolution)

# Create a baseline latitude thermal gradient (15°C to 25°C)
base_sst = np.linspace(15, 25, lat_grid)[:, None] + np.random.normal(0, 0.2, (lat_grid, lon_grid))

# Generate daily ocean surface grids over time
time_series = []
for day in range(num_days):
    daily_noise = np.random.normal(0, 0.3, (lat_grid, lon_grid))
    
    # Inject a localized high-temperature anomaly simulating a Marine Heatwave (MHW)
    if 10 <= day <= 20:
        daily_noise[12:20, 12:20] += 4.5  # Local thermal anomaly bump (+4.5°C)
    
    # Combine baseline gradient, temporal oscillation, and daily noise
    daily_sst = base_sst + 1.5 * np.sin(day / 5.0) + daily_noise
    time_series.append(daily_sst)

# Convert Python list to 3D NumPy array
sst_numpy = np.array(time_series, dtype=np.float32)

print("--- Cell 1 Output ---")
print(f"NumPy Raw Array Shape (Time, Lat, Lon): {sst_numpy.shape}")
print(f"Minimum Temperature: {sst_numpy.min():.2f} °C")
print(f"Maximum Temperature: {sst_numpy.max():.2f} °C")

--- Cell 1 Output ---
NumPy Raw Array Shape (Time, Lat, Lon): (30, 32, 32)
Minimum Temperature: 12.54 °C
Maximum Temperature: 27.68 °C


## Cell 2: Converting NumPy Arrays to PyTorch Tensors & Spatiotemporal Slicing

### PyTorch Tensor Layout for 2D Convolutions (`nn.Conv2d`)
Deep learning architectures for spatiotemporal forecasting (such as ConvLSTM or 2D CNNs) expect input data structured as **4D tensors**:

$$\text{Shape: } [\text{Batch\_Size}, \text{Channels}, \text{Height (Latitude)}, \text{Width (Longitude)}]$$

In our oceanographic context:
* **Batch Size ($N$):** Number of independent input samples.
* **Channels ($C$):** Historical temporal window (e.g., 5 consecutive days of observations).
* **Height ($H$) & Width ($W$):** Spatial dimensions representing the oceanic grid resolution ($32 \times 32$).

### Supervised Learning via Sliding Window
We construct training pairs $(X, Y)$ using a sequence length of $N_{steps} = 5$:
1. **Input ($X$):** Historical SST sequence from day $t$ to day $t+4$ (Shape: `[5, 32, 32]`).
2. **Target ($Y$):** Sea Surface Temperature on day $t+5$ (Shape: `[1, 32, 32]`).

In [5]:
"""
Module 01 - Cell 2: Converting to PyTorch Tensors & Spatiotemporal Slicing
----------------------------------------------------------------------------
This cell converts the raw 3D NumPy array into structured 4D PyTorch tensors (X, Y)
using a sliding window approach for supervised time-series forecasting.
"""

import torch

# 1. Convert NumPy float32 array to a PyTorch Tensor
sst_tensor = torch.from_numpy(sst_numpy)

# Set the sliding window length (number of historical days used as input)
sequence_length = 5  

def create_spatiotemporal_batches(tensor, seq_len):
    """
    Slices a 3D spatio-temporal tensor (Time, Lat, Lon) into overlapping input 
    sequences (X) and target days (Y) for supervised deep learning.
    
    Args:
        tensor (torch.Tensor): Input tensor of shape (Total_Days, Lat, Lon).
        seq_len (int): Historical days per sample.
        
    Returns:
        X (torch.Tensor): Model inputs of shape (N_Samples, Seq_Len, Lat, Lon).
        Y (torch.Tensor): Targets of shape (N_Samples, 1, Lat, Lon).
    """
    X_list, Y_list = [], []
    total_steps = tensor.shape[0]
    
    # Iterate over time series to create overlapping sliding windows
    for i in range(total_steps - seq_len):
        # Slice sequence of N historical days -> Shape: (Seq_Len, Lat, Lon)
        x_sample = tensor[i : i + seq_len]          
        
        # Slice immediate next day as target and insert channel dimension -> Shape: (1, Lat, Lon)
        y_sample = tensor[i + seq_len].unsqueeze(0) 
        
        X_list.append(x_sample)
        Y_list.append(y_sample)
        
    # Stack list of tensors into unified 4D dataset tensors
    return torch.stack(X_list), torch.stack(Y_list)

# Generate spatio-temporal dataset tensors
X_data, Y_data = create_spatiotemporal_batches(sst_tensor, sequence_length)

print("--- Cell 2 Output ---")
print(f"PyTorch Input Tensor X Shape (Samples, Time_Channels, Lat, Lon): {X_data.shape}")
print(f"PyTorch Target Tensor Y Shape (Samples, Target_Channel, Lat, Lon): {Y_data.shape}")

--- Cell 2 Output ---
PyTorch Input Tensor X Shape (Samples, Time_Channels, Lat, Lon): torch.Size([25, 5, 32, 32])
PyTorch Target Tensor Y Shape (Samples, Target_Channel, Lat, Lon): torch.Size([25, 1, 32, 32])


## Cell 3: Implementing PyTorch `Dataset` and `DataLoader`

### Why do we need `Dataset` and `DataLoader`?
When training deep learning models on spatio-temporal marine data (`X_data` and `Y_data`), we cannot simply pass all 25 samples into the neural network at once. We need structured iteration:
* **`Dataset`**: An abstract class in PyTorch that wraps our tensors and defines how individual samples are indexed and retrieved (`__len__` and `__getitem__`).
* **`DataLoader`**: A utility that wraps the `Dataset` to automatically handle **mini-batching** (grouping samples into batches, e.g., batch size of 4), **shuffling** (randomizing data order to prevent model bias), and parallel loading.

In [6]:
"""
Module 01 - Cell 3: Custom PyTorch Dataset & DataLoader
------------------------------------------------------------
Wraps our spatio-temporal tensors into PyTorch Dataset and DataLoader 
to enable mini-batching and shuffling for training.
"""

import torch
from torch.utils.data import Dataset, DataLoader

class MarineHeatwaveDataset(Dataset):
    """
    Custom PyTorch Dataset for Spatio-Temporal Marine Heatwave (MHW) forecasting.
    """
    def __init__(self, X_tensor, Y_tensor):
        self.X = X_tensor
        self.Y = Y_tensor
        
    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.X)
        
    def __getitem__(self, idx):
        """Retrieves a single sample pair (X, Y) by index."""
        return self.X[idx], self.Y[idx]

# 1. Instantiate the custom dataset using our 4D tensors from Cell 2
mhw_dataset = MarineHeatwaveDataset(X_data, Y_data)

# 2. Configure the DataLoader
batch_size = 4
mhw_dataloader = DataLoader(
    dataset=mhw_dataset,
    batch_size=batch_size,
    shuffle=True,       # Shuffle data at every epoch to improve learning
    drop_last=False     # Keep remaining samples if total doesn't divide evenly
)

# 3. Verification: Inspect the first batch from the DataLoader
print("--- Cell 3 Output ---")
print(f"Total samples in Dataset: {len(mhw_dataset)}")
print(f"Total mini-batches in DataLoader: {len(mhw_dataloader)} (batch_size={batch_size})")

for batch_idx, (batch_x, batch_y) in enumerate(mhw_dataloader):
    print(f"\n[Inspection of Batch {batch_idx + 1}]")
    print(f"  -> Batch X shape (Batch, Time_Channels, Lat, Lon): {batch_x.shape}")
    print(f"  -> Batch Y shape (Batch, Target_Channel, Lat, Lon): {batch_y.shape}")
    break  # We only inspect the first batch for verification

--- Cell 3 Output ---
Total samples in Dataset: 25
Total mini-batches in DataLoader: 7 (batch_size=4)

[Inspection of Batch 1]
  -> Batch X shape (Batch, Time_Channels, Lat, Lon): torch.Size([4, 5, 32, 32])
  -> Batch Y shape (Batch, Target_Channel, Lat, Lon): torch.Size([4, 1, 32, 32])


## 📝 Module 01 - Executive Summary: Data Foundations & Pipeline

In this first module, we established the fundamental data engineering pipeline required to forecast Marine Heatwaves (MHW) using PyTorch:

1. **Synthetic SST Generation:** 
   - Created a 3D NumPy array `(Time, Lat, Lon)` simulating Sea Surface Temperature anomalies over a $32 \times 32$ spatial grid across 30 days.
2. **Temporal Sliding Window:**
   - Transformed the time series into supervised training pairs $(X, Y)$ using a historical window of 5 days ($N_{steps} = 5$) to predict the immediate next day ($t+5$).
3. **4D Tensors for Convolutions (`nn.Conv2d`):**
   - Structured inputs into PyTorch's standard format: `[Batch_Size, Channels, Height, Width]`, treating historical steps as temporal channels.
4. **Efficient Pipeline (`Dataset` & `DataLoader`):**
   - Implemented a custom subclass of `torch.utils.data.Dataset` wrapped in a `DataLoader` configured with mini-batches (`batch_size = 4`) and active shuffling (`shuffle=True`), ensuring robust data delivery for neural network training.